In [1]:
import os

folder_path = r"C:\Users\prakh\Downloads\nlp\dataset\Finance"

if os.path.exists(folder_path):
    print("Folder found! Here is what is inside:")
    print(os.listdir(folder_path))
else:
    print("Folder STILL NOT FOUND. Double-check the path name in your Windows File Explorer.")

Folder found! Here is what is inside:
['test.json', 'train.json']


In [2]:
import os
import gc
import ast
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import shap
import matplotlib.pyplot as plt

# ==========================================
# Phase 1: Load SEntFiN-v1.1 & Flatten Dictionary
# ==========================================
print("--- Phase 1: Loading & Flattening SEntFiN Dataset ---")
DATA_FILE = r"C:\Users\prakh\Downloads\nlp\SEntFiN-v1.1.csv"

# Load the raw CSV
raw_df = pd.read_csv(DATA_FILE)

# The SEntFiN dataset has a 'Title' column and a 'Decisions' column containing dicts
processed_data = []
for _, row in raw_df.iterrows():
    # Fallback to lowercase if columns are named differently
    headline = row.get('Title', row.get('title', ''))
    decisions_str = row.get('Decisions', row.get('decisions', '{}'))
    
    try:
        # Convert the string '{"Nvidia": "positive"}' into a real Python dictionary
        decisions_dict = ast.literal_eval(decisions_str)
        
        # Create a new row for EVERY company mentioned in the headline
        for aspect, sentiment in decisions_dict.items():
            processed_data.append({
                "headline": headline,
                "aspect": aspect,
                "sentiment": sentiment
            })
    except (ValueError, SyntaxError):
        continue # Skip corrupted rows

# Convert to clean DataFrame
flat_df = pd.DataFrame(processed_data)

# Map labels to 0, 1, 2
encoder = LabelEncoder()
flat_df['sentiment_encoded'] = encoder.fit_transform(flat_df['sentiment'].astype(str).str.lower())
target_names = [str(cls).title() for cls in encoder.classes_]

# Split into Train and Test (80% Train, 20% Test)
train_df, test_df = train_test_split(flat_df, test_size=0.2, random_state=42, stratify=flat_df['sentiment_encoded'])

y_train = train_df['sentiment_encoded'].values
y_test = test_df['sentiment_encoded'].values

print(f"Dataset Flattened! Total Extractable Signals: {flat_df.shape[0]}")
print(f"Train Shape: {train_df.shape[0]} | Test Shape: {test_df.shape[0]}")
print(f"Detected Target Classes: {target_names}")

# ==========================================
# Phase 2: Initialize LLM Core
# ==========================================
print("\n--- Phase 2: Mounting T5 Architecture ---")
MODEL_NAME = "amphora/FinABSA"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()
model.to(device)

# ==========================================
# Phase 3: Latent Feature Extractor
# ==========================================
def extract_latent_features(headlines, entities, batch_size=16):
    """
    Passes text through the T5 Encoder, calculates the masked center-of-gravity, 
    and returns a 1024-D numeric signature. Bypasses text generation for speed.
    """
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(headlines), batch_size), desc="Extracting 1024-D Signals"):
            batch_heads = headlines[i : i+batch_size]
            batch_ents = entities[i : i+batch_size]
            
            # Anchor the aspect into the text
            formatted_batch = []
            for head, ent in zip(batch_heads, batch_ents):
                target_text = str(head).replace(str(ent), "[TGT]")
                if "[TGT]" not in target_text:
                    target_text = f"[TGT] {head}"
                formatted_batch.append(f"Sentiment Analysis: {target_text}")
            
            # Tokenize & push to device
            encoded = tokenizer(formatted_batch, padding=True, truncation=True, max_length=64, return_tensors='pt')
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)
            
            # ENCODER ONLY (Bypass slow text generation)
            encoder_outputs = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
            
            # Masked Mean Pooling (ignores padding 0s)
            mask_expanded = attention_mask.unsqueeze(-1).expand(encoder_outputs.last_hidden_state.size()).float()
            sum_embeddings = torch.sum(encoder_outputs.last_hidden_state * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            mean_pooled = sum_embeddings / sum_mask
            
            all_embeddings.append(mean_pooled.cpu().numpy())
            
            # Memory flush
            del input_ids, attention_mask, encoder_outputs, mask_expanded, sum_embeddings, mean_pooled
            torch.cuda.empty_cache()
            
    return np.vstack(all_embeddings)

print("\n--- Phase 3: Executing Feature Extraction ---")
print("Processing Training Split...")
X_train = extract_latent_features(train_df['headline'].tolist(), train_df['aspect'].tolist())

print("Processing Testing Split...")
X_test = extract_latent_features(test_df['headline'].tolist(), test_df['aspect'].tolist())

# Purge the LLM from memory
del model
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# Phase 4: XGBoost Financial Classifier
# ==========================================
print("\n--- Phase 4: Training Hybrid Tree Model ---")
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    tree_method='hist', # High-speed dense matrix calculation
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("\n--- Final Evaluation ---")
y_pred = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=target_names))
print(f"Hybrid Pipeline Accuracy: {accuracy_score(y_test, y_pred):.4f}")


# Save the trained XGBoost model and LabelEncoder
import joblib
xgb_model.save_model('xgb_financial_model.json')
joblib.dump(encoder, 'label_encoder.pkl')
print('Models saved successfully to disk.')


--- Phase 1: Loading & Flattening SEntFiN Dataset ---
Dataset Flattened! Total Extractable Signals: 14409
Train Shape: 11527 | Test Shape: 2882
Detected Target Classes: ['Negative', 'Neutral', 'Positive']

--- Phase 2: Mounting T5 Architecture ---
Using compute device: cpu


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/amphora/FinABSA/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].



--- Phase 3: Executing Feature Extraction ---
Processing Training Split...


Extracting 1024-D Signals:   0%|          | 0/721 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [3]:
# ==========================================
# Phase 6: Live Inference Testing
# ==========================================
print("\n--- Phase 6: Live Inference Engine ---")

# 1. Reload the T5 model (since we deleted it in Phase 3 to save RAM)
try:
    model
except NameError:
    print("Reloading T5 Encoder for new predictions...")
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.eval()
    model.to(device)


# Reload the XGBoost model and LabelEncoder if not in memory
import joblib
import xgboost as xgb
try:
    xgb_model
    encoder
except NameError:
    print('Reloading XGBoost model and encoder from disk...')
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model('xgb_financial_model.json')
    encoder = joblib.load('label_encoder.pkl')
def predict_custom_sentence(headline: str, aspect: str):
    """
    Takes a custom headline and target company, extracts the 1024-D signature, 
    and predicts the financial sentiment using the trained XGBoost model.
    """
    # Format the text
    target_text = str(headline).replace(str(aspect), "[TGT]")
    if "[TGT]" not in target_text:
        target_text = f"[TGT] {headline}"
    formatted_text = f"Sentiment Analysis: {target_text}"
    
    # Tokenize
    encoded = tokenizer([formatted_text], padding=True, truncation=True, max_length=64, return_tensors='pt')
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    
    # Extract 1024-D Latent Signature
    with torch.no_grad():
        encoder_outputs = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        mask_expanded = attention_mask.unsqueeze(-1).expand(encoder_outputs.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(encoder_outputs.last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        latent_features = (sum_embeddings / sum_mask).cpu().numpy()
        
    # Predict using the trained XGBoost tree
    probabilities = xgb_model.predict_proba(latent_features)[0]
    pred_class_idx = np.argmax(probabilities)
    confidence = probabilities[pred_class_idx]
    
    # Map the numeric prediction (e.g., 2) back to the word (e.g., "Positive")
    pred_label = encoder.inverse_transform([pred_class_idx])[0]
    
    return pred_label.title(), confidence

# ==========================================
# Run Your Custom Tests
# ==========================================
my_test_cases = [
    # (Headline, Target Entity)
    ("Nvidia smashes Q3 earnings expectations, sending tech stocks soaring.", "Nvidia"),
    ("Despite strong revenue, Nvidia faces severe supply chain delays in Taiwan.", "Nvidia"),
    ("Apple announces new iPhone, but Google's stock remains completely flat.", "Google"),
    ("Federal Reserve hikes interest rates unexpectedly, causing market panic.", "Federal Reserve")
]

print("\n--- Custom Test Results ---")
for headline, aspect in my_test_cases:
    sentiment, confidence = predict_custom_sentence(headline, aspect)
    print(f"\nHeadline: '{headline}'")
    print(f"Target:   {aspect}")
    print(f"Signal:   {sentiment} (Confidence: {confidence:.2%})")


--- Phase 6: Live Inference Engine ---
Reloading XGBoost model and encoder from disk...

--- Custom Test Results ---

Headline: 'Nvidia smashes Q3 earnings expectations, sending tech stocks soaring.'
Target:   Nvidia
Signal:   Positive (Confidence: 95.40%)

Headline: 'Despite strong revenue, Nvidia faces severe supply chain delays in Taiwan.'
Target:   Nvidia
Signal:   Negative (Confidence: 99.05%)

Headline: 'Apple announces new iPhone, but Google's stock remains completely flat.'
Target:   Google
Signal:   Neutral (Confidence: 97.94%)

Headline: 'Federal Reserve hikes interest rates unexpectedly, causing market panic.'
Target:   Federal Reserve
Signal:   Negative (Confidence: 69.61%)


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create the UI elements
headline_input = widgets.Text(
    description='Headline:', 
    placeholder='e.g., Nvidia stock plummets despite record revenues.',
    layout=widgets.Layout(width='80%')
)

aspect_input = widgets.Text(
    description='Target:', 
    placeholder='e.g., Nvidia',
    layout=widgets.Layout(width='40%')
)

analyze_button = widgets.Button(
    description='Analyze Sentiment', 
    button_style='success', # Makes the button green
    icon='bolt'
)

output_area = widgets.Output()

# 2. Define what happens when you click the button
def on_analyze_click(b):
    with output_area:
        clear_output() # Clears the previous result
        h = headline_input.value.strip()
        a = aspect_input.value.strip()
        
        if not h or not a:
            print("⚠️ Please enter both a headline and a target entity.")
            return
            
        print(f"Extracting 1024-D signature for: '{a}'...")
        sentiment, confidence = predict_custom_sentence(h, a)
        
        # Color coding the output
        color = "green" if sentiment.lower() == "positive" else "red" if sentiment.lower() == "negative" else "gray"
        
        print("\n--- HYBRID MODEL RESULT ---")
        print(f"Headline: {h}")
        print(f"Target:   {a}")
        print(f"Signal:   \033[1;3{1 if color=='red' else 2 if color=='green' else 0}m{sentiment.upper()}\033[0m")
        print(f"Conf:     {confidence:.2%}")

# 3. Link the button to the function and display the UI
analyze_button.on_click(on_analyze_click)
display(headline_input, aspect_input, analyze_button, output_area)

Text(value='', description='Headline:', layout=Layout(width='80%'), placeholder='e.g., Nvidia stock plummets d…

Text(value='', description='Target:', layout=Layout(width='40%'), placeholder='e.g., Nvidia')

Button(button_style='success', description='Analyze Sentiment', icon='bolt', style=ButtonStyle())

Output()

In [6]:
# In Untitled.ipynb after your train_test_split:
test_df.to_csv("official_test_set.csv", index=False)

In [18]:
# Quick example of what true XAI code looks like for your model:
import shap

# Initialize the SHAP explainer for your fine-tuned model
explainer = shap.Explainer(model, tokenizer)
shap_values = explainer(["Nvidia soars"])

# This generates a visualization showing the impact of each word
shap.plots.text(shap_values[0])

C:\Users\prakh\AppData\Roaming\Python\Python310\site-packages\transformers\generation\utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


  0%|          | 0/42 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:17, 17.51s/it]               


In [13]:
import torch
import numpy as np
import pandas as pd
import ast
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display

# 1. LOAD THE RAW CSV PROPERLY
file_path = r"C:\Users\prakh\Downloads\nlp\SEntFiN-v1.1.csv" 
raw_df = pd.read_csv(file_path)

# Sample 100 raw articles to keep it fast
raw_df = raw_df.sample(n=100, random_state=42).reset_index(drop=True)

# 2. FLATTEN THE DATA (Mini-Phase 1)
# The raw dataset has a 'Decisions' column with strings like '{"Apple": "positive", "Nvidia": "negative"}'
flattened_records = []
for _, row in raw_df.iterrows():
    text = row['Title']
    # Convert the string dictionary into a real Python dictionary
    try:
        decisions = ast.literal_eval(row['Decisions'])
    except:
        continue 
        
    for entity, sentiment in decisions.items():
        # Inject the [TGT] tag right after the entity name so the T5 Encoder knows what to focus on
        tagged_text = text.replace(entity, f"{entity} [TGT]")
        flattened_records.append({"title": tagged_text, "label": sentiment.lower()})

# Create the final dataframe that the evaluation loop expects
dataset_df = pd.DataFrame(flattened_records)
print(f"Flattened 100 raw articles into {len(dataset_df)} individual entity signals.\n")

# 3. PREPARE THE LABELS
preds = []
labels = dataset_df["label"].tolist() 

# Map XGBoost integer outputs back to strings
label_map = {0: "negative", 1: "neutral", 2: "positive"} 
valid_labels = ["positive", "negative", "neutral"]

# Fix target mismatch: if the JSON holds integers, turn them into strings
if isinstance(labels[0], (int, np.integer)):
    labels = [label_map.get(lbl, "neutral") for lbl in labels]

print("Evaluating hybrid T5-Encoder + XGBoost pipeline...")
with torch.no_grad():
    for i, row in dataset_df.iterrows():
        # Tokenize the input text (which now has the [TGT] tag)
        inputs = tokenizer(row["title"], return_tensors='pt', truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Extract embeddings strictly from the Encoder
        if hasattr(model, 'encoder'):
            outputs = model.encoder(**inputs)
        else:
            outputs = model(**inputs)
            
        # Get the 1024-D embedding array
        embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        
        # Wrap the embedding in a DataFrame so XGBoost recognizes the feature names
        feature_names = [f"Latent_Dim_{j}" for j in range(embedding.shape[1])]
        embedding_df = pd.DataFrame(embedding, columns=feature_names)
        
        # Predict using XGBoost
        pred_idx = xgb_model.predict(embedding_df)[0]
        
        # Map back to string
        extracted_val = label_map.get(pred_idx, "neutral")
        preds.append(extracted_val)

# 4. PRINT EVALUATION RESULTS
print("\n=== Distil (T5-Encoder + XGBoost) Evaluation ===\n")
print(classification_report(labels, preds, target_names=valid_labels))

print("\nConfusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(labels, preds, labels=valid_labels), 
    index=valid_labels, 
    columns=valid_labels
))

Flattened 100 raw articles into 143 individual entity signals.

Evaluating hybrid T5-Encoder + XGBoost pipeline...

=== Distil (T5-Encoder + XGBoost) Evaluation ===

              precision    recall  f1-score   support

    positive       1.00      0.90      0.95        51
    negative       0.89      0.94      0.91        50
     neutral       0.89      0.93      0.91        42

    accuracy                           0.92       143
   macro avg       0.92      0.92      0.92       143
weighted avg       0.93      0.92      0.92       143


Confusion Matrix:


,positive,negative,neutral
positive,39,0,3
negative,2,46,3
neutral,3,0,47


In [21]:
import torch
import numpy as np
import shap
import matplotlib.pyplot as plt

def explain_any_sentence(custom_headline, target_aspect):
    """
    Bridges the gap: Text -> T5 Encoder -> 1024D Vector -> XGBoost -> SHAP
    """
    # 1. PRE-PROCESSING: The [TGT] Anchor trick
    # Replace the aspect with [TGT] so the model knows who to look at
    target_text = custom_headline.replace(target_aspect, "[TGT]")
    if "[TGT]" not in target_text:
        target_text = f"[TGT] {custom_headline}"
    prompt = f"Sentiment Analysis: {target_text}"

    # 2. FEATURE EXTRACTION: Get the 1024D signature
    model.to(device)
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
    
    with torch.no_grad():
        # Get the hidden states from the T5 Encoder
        encoder_outputs = model.encoder(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        # Mean Pooling to get a single 1024-D vector
        embeddings = encoder_outputs.last_hidden_state.mean(dim=1).cpu().numpy()

    # 3. PREDICTION
    pred_idx = xgb_model.predict(embeddings)[0]
    confidence = xgb_model.predict_proba(embeddings)[0][pred_idx]
    label = target_names[pred_idx]

    print(f"\nHeadline: {custom_headline}")
    print(f"Target: {target_aspect}")
    print(f"Result: {label} ({confidence*100:.2f}% confidence)")

    # 4. SHAP ANALYSIS: Explain the 1024 Dimensions
    explainer = shap.TreeExplainer(xgb_model)
    # Calculate SHAP for this specific vector
    shap_vals = explainer.shap_values(embeddings)

    # Handle SHAP multi-class list output
    if isinstance(shap_vals, list):
        shap_vals_to_plot = shap_vals[pred_idx]
    else:
        # For newer versions where it returns [samples, features, classes]
        shap_vals_to_plot = shap_vals[0, :, pred_idx] if shap_vals.ndim == 3 else shap_vals[0]

    # 5. VISUALIZE: Force Plot
    # This shows which dimensions 'pushed' the sentiment to the predicted class
    shap.initjs()
    return shap.force_plot(
        explainer.expected_value[pred_idx] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
        shap_vals_to_plot,
        feature_names=[f"Dim_{i}" for i in range(embeddings.shape[1])]
    )

# --- TEST IT HERE ---
explain_any_sentence("Nvidia shares skyrocket after massive earnings beat", "Nvidia")


Headline: Nvidia shares skyrocket after massive earnings beat
Target: Nvidia
Result: Positive (99.24% confidence)


In [20]:
def check_custom_headline(custom_headline, target_aspect):
    # 1. Transform the text into the same 1024-D format the model learned
    # This is the "Bridge" between your words and XGBoost's numbers
    custom_feature_vector = extract_latent_features([custom_headline], [target_aspect], batch_size=1)
    
    # 2. Get the Prediction
    pred_idx = xgb_model.predict(custom_feature_vector)[0]
    prediction = target_names[pred_idx]
    
    print(f"\n--- Analysis for: '{custom_headline}' ---")
    print(f"Target Aspect: {target_aspect}")
    print(f"Predicted Sentiment: {prediction}")

    # 3. Generate the SHAP Force Plot for THIS specific input
    # This shows exactly which 'features' (latent dimensions) moved the needle
    explainer = shap.TreeExplainer(xgb_model)
    instance_shap = explainer.shap_values(custom_feature_vector)
    
    # Select the SHAP values for the predicted class
    if isinstance(instance_shap, list):
        shap_for_pred = instance_shap[pred_idx]
    else:
        # For newer versions of SHAP/XGBoost, it might be [samples, features, classes]
        # or just [samples, features]. Let's slice carefully:
        shap_for_pred = instance_shap[0] if instance_shap.ndim == 2 else instance_shap[0, :, pred_idx]

    # Initialize JS for the beautiful interactive plot
    shap.initjs()
    return shap.force_plot(
        explainer.expected_value[pred_idx] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
        shap_for_pred,
        feature_names=[f"Dim_{i}" for i in range(custom_feature_vector.shape[1])]
    )

# --- TRY IT HERE ---
check_custom_headline("Reliance Jio announces record-breaking profits this quarter", "Reliance Jio")

Extracting 1024-D Signals: 100%|██████████| 1/1 [00:00<00:00,  4.68it/s]



--- Analysis for: 'Reliance Jio announces record-breaking profits this quarter' ---
Target Aspect: Reliance Jio
Predicted Sentiment: Positive
